In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_community.tools.playwright.utils import create_sync_playwright_browser
from langchain_community.tools.playwright import (
    ClickTool, NavigateTool, ExtractTextTool, ExtractHyperlinksTool, GetElementsTool, NavigateBackTool, CurrentWebPageTool
)
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.graph import START, END
from langgraph.checkpoint.memory import MemorySaver
import asyncio
import os


In [ ]:
load_dotenv(override=True)

In [ ]:
# First define a structured output

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if more input is needed from the user, or clarifications, or the assistant is stuck")


In [ ]:
# The state

class State(TypedDict):
    
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

In [ ]:
# Define stock market tool using screener.in

from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def get_stock_details(company_name: str) -> str:
    """
    Get stock market details for a company from screener.in.
    Pass the company name (e.g., 'TCS', 'Reliance', 'HDFC Bank').
    Returns detailed stock information including price, fundamentals, and ratios.
    """
    try:
        # Search for the company on screener.in
        search_url = f"https://www.screener.in/search/?q={company_name.replace(' ', '+')}"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(search_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Try to extract company information
        # Look for company name, price, market cap, PE ratio, etc.
        result = f"Fetching stock data for {company_name} from screener.in...\n"
        
        # Extract key information if available
        company_links = soup.find_all('a', class_='company')
        if company_links:
            for link in company_links[:3]:  # Get top 3 results
                company = link.text.strip()
                url = link.get('href', '')
                result += f"  • {company}\n"
        
        if len(company_links) == 0:
            result += "No results found. Try searching with ticker symbol (e.g., 'TCS', 'RELIANCE').\n"
            result += f"Visit: https://www.screener.in/search/?q={company_name.replace(' ', '+')}"
        
        return result
        
    except requests.exceptions.RequestException as e:
        return f"Error fetching data from screener.in: {str(e)}\n" \
               f"Try visiting https://www.screener.in/search/?q={company_name.replace(' ', '+')}"

# Create tools list
tools = [get_stock_details]

print(f"✓ Stock market tool defined")
print(f"  - Tool: {tools[0].name}")
print(f"  - Description: {tools[0].description}")

In [ ]:
from langchain_groq import ChatGroq  # make sure installed

# Get API key
api_key = os.getenv("GROQ_API_KEY")

print(f"\nAPI Key loaded: {bool(api_key)}")

if api_key:
    print(f"✓ API Key found (first 20 chars): {api_key[:20]}...")

    try:
        # Worker LLM
        worker_llm = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0,
            api_key=api_key
        )

        worker_llm_with_tools = worker_llm.bind_tools(tools)

        # Evaluator LLM
        evaluator_llm = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0,
            api_key=api_key
        )

        evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

        print("\n✓ Groq LLMs initialized successfully!")

    except Exception as e:
        print(f"\nError initializing LLMs: {e}")

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    system_message = f"""..."""

    if state.get("feedback_on_work"):
        system_message += f"""..."""

    messages = list(state["messages"])  

    found_system_message = False
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True

    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    response = worker_llm_with_tools.invoke(messages)

    return {
        "messages": [response],  
    }

In [ ]:
def worker_router(state: State) -> str:
    messages = state.get("messages", [])
    
    if not messages:
        return "evaluator"
    
    last_message = messages[-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    return "evaluator"

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def format_conversation(messages: List[Any]) -> str:
    conversation = "Conversation history:\n\n"
    
    for message in messages:
        content = getattr(message, "content", "") or "[No content]"
        
        if isinstance(message, HumanMessage):
            conversation += f"User: {content}\n"
        
        elif isinstance(message, AIMessage):
            if hasattr(message, "tool_calls") and message.tool_calls:
                conversation += "Assistant: [Tool call requested]\n"
            else:
                conversation += f"Assistant: {content}\n"
        
        elif isinstance(message, ToolMessage):
            conversation += f"Tool Output: {content}\n"
    
    return conversation

In [ ]:
def evaluator(state: State) -> State:
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""
    
    user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.
"""
    if state["feedback_on_work"]:
        user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
        user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."
    
    evaluator_messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

    eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
    new_state = {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback on this answer: {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed
    }
    return new_state

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    success = state.get("success_criteria_met", False)
    user_input_needed = state.get("user_input_needed", False)

    if success or user_input_needed:
        return "END"
    
    return "worker"

In [ ]:
# Set up Graph Builder with State
graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

# Add edges
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
graph_builder.add_edge(START, "worker")

# Compile the graph
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage

async def process_message(message, success_criteria, history, thread):
    try:
        history = history or []

        config = {"configurable": {"thread_id": thread}}

        state = {
            "messages": [HumanMessage(content=message)],
            "success_criteria": success_criteria,
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False
        }

        result = await graph.ainvoke(state, config=config)

        messages = result.get("messages", [])

        reply = ""
        feedback = ""

        if len(messages) >= 2:
            reply = str(messages[-2].content)

        if len(messages) >= 1:
            feedback = str(messages[-1].content)

        assistant_text = reply + "\n\n🧠 Feedback:\n" + feedback

        # ✅ STRICT FORMAT (IMPORTANT)
        history.append({"role": "user", "content": str(message)})
        history.append({"role": "assistant", "content": str(assistant_text)})

        return history

    except Exception as e:
        # ✅ ALWAYS RETURN VALID FORMAT EVEN ON ERROR
        history = history or []
        history.append({"role": "user", "content": str(message)})
        history.append({"role": "assistant", "content": f"❌ Error: {str(e)}"})
        return history
    

def reset_sync():
 return "", "", [], make_thread_id()

In [ ]:
import asyncio
import gradio as gr


# Sync wrapper (safe for notebook + scripts)
def process_message_sync(message, success_criteria, history, thread):
    try:
        return asyncio.run(process_message(message, success_criteria, history, thread))
    except RuntimeError:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(
            process_message(message, success_criteria, history, thread)
        )


# Reset function
def reset_sync():
    return "", "", [], make_thread_id()


with gr.Blocks() as demo:
    gr.Markdown("## 📈 Stock Market Assistant (LangGraph + Groq)")

    thread = gr.State(make_thread_id())

    # ✅ Chatbot now uses message format
    chatbot = gr.Chatbot(label="Sidekick", height=300)

    # Inputs
    message = gr.Textbox(
        placeholder="Your request to your sidekick",
        show_label=False
    )

    success_criteria = gr.Textbox(
        placeholder="What are your success criteria?",
        show_label=False
    )

    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    # ✅ Button click
    go_button.click(
        fn=process_message_sync,
        inputs=[message, success_criteria, chatbot, thread],
        outputs=chatbot
    )

    # ✅ Enter key (only message box)
    message.submit(
        fn=process_message_sync,
        inputs=[message, success_criteria, chatbot, thread],
        outputs=chatbot
    )

    # ❌ REMOVE this (causes unwanted triggers)
    # success_criteria.submit(...)

    # ✅ Reset
    reset_button.click(
        fn=reset_sync,
        inputs=[],
        outputs=[message, success_criteria, chatbot, thread]
    )


# ✅ Theme moved here (Gradio 6)
demo.launch(theme=gr.themes.Default(primary_hue="emerald"))